# Importation bibliothèque

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Sites à web scrapper

## Sirius-upvm.net
Permet d'obtenir un dictionnaire des pays (nom & code)

### Téléchargement de la page et 'parsing' du HTML

In [12]:
url = "https://www.sirius-upvm.net/doc/usuels/iso3166.html"
response = requests.get(url)
response.encoding = 'utf-8'  # important pour les accents (é, è, î...)
soup = BeautifulSoup(response.text, "html.parser")

### Transformation sous forme de dataframe

In [13]:
# Cette page a plusieurs tableaux, on veut ceux avec 4 colonnes (Pays, 2L, 3L, num)
tous_les_tableaux = soup.find_all("table")

data = []

for tableau in tous_les_tableaux:
    lignes = tableau.find_all("tr")
    for ligne in lignes:
        cellules = ligne.find_all("td")
        # On garde uniquement les lignes avec exactement 4 colonnes
        if len(cellules) == 4:
            pays        = cellules[0].text.strip()
            code_2L     = cellules[1].text.strip()
            code_3L     = cellules[2].text.strip()
            code_num    = cellules[3].text.strip()
            # On ignore la ligne d'en-tête (Pays, code à 2 lettres...)
            if pays.upper() != "PAYS" and pays != "Pays ou territoire":
                data.append([pays, code_2L, code_3L, code_num])

df = pd.DataFrame(data, columns=["pays", "code_2L", "code_3L", "code_numerique"])

# Nettoyer les doublons éventuels
df_index_pays = df.drop_duplicates()
df_index_pays.head()

,pays,code_2L,code_3L,code_numerique
0,ALLEMAGNE,DE,DEU,276
1,AUTRICHE,AT,AUT,040
2,BELGIQUE,BE,BEL,056
3,BULGARIE,BG,BGR,100
4,CHYPRE,CY,CYP,196


### Extraction du dataframe

In [14]:
df_index_pays.to_csv(r"C:\Users\Utilisateur\Desktop\ProjetCertif\ProjetCertif\DATA\processed\iso3166.csv", index=False)

## Numbeo.com 
Pour les indices de qualités de vie/pays

### Téléchargement de la page et 'parsing' du HTML

In [15]:
url = "https://fr.numbeo.com/qualit%C3%A9-de-vie/classements-par-pays?title=2024-mid"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

### Récupération des données

In [16]:
# On cible directement le tableau par son id
tableau = soup.find("table", id="t2")

# Extraire les en-têtes
colonnes = [th.get_text(strip=True) for th in tableau.find("thead").find_all("th")]
print("Colonnes :", colonnes)

# Extraire les lignes
data = []
for ligne in tableau.find("tbody").find_all("tr"):
    cellules = ligne.find_all("td")
    if cellules:
        valeurs = [cellule.get_text(strip=True) for cellule in cellules]
        data.append(valeurs)

df = pd.DataFrame(data, columns=colonnes)

# Convertir les colonnes numériques
for col in df.columns:
    if col not in ["Classement", "Pays"]:
        df[col] = df[col].str.replace(",", ".").astype(float, errors="ignore")

df.head()

Colonnes : ['Classement', 'Pays', 'Indice de Qualité de Vie', "Indice du Pouvoir d'Achat", 'Indice de Sécurité', 'Indice des soins de santé', 'Indice du Coût de la Vie', "Rapport Prix de l'immobilier/Revenu", 'Indice du Temps de Trajet dans Trafic', 'Indice de pollution', 'Indice Climatique']


,Classement,Pays,Indice de Qualité de Vie,Indice du Pouvoir d'Achat,Indice de Sécurité,Indice des soins de santé,Indice du Coût de la Vie,Rapport Prix de l'immobilier/Revenu,Indice du Temps de Trajet dans Trafic,Indice de pollution,Indice Climatique
0,,Luxembourg,219.32,182.53,65.72,75.33,62.43,8.88,27.18,23.27,82.62
1,,Pays-Bas,207.48,124.91,73.58,79.28,63.12,7.69,23.48,21.38,87.01
2,,Danemark,205.57,127.20,73.89,78.47,72.27,6.58,27.81,20.75,83.74
3,,Oman,204.30,139.80,81.77,65.11,42.37,2.88,20.96,35.04,67.22
4,,Suisse,203.95,158.68,73.91,71.73,101.10,10.40,32.93,23.31,82.04


### Extraction du dataframe

In [17]:
df_life_quality = df
df_life_quality.to_csv(r"C:\Users\Utilisateur\Desktop\ProjetCertif\ProjetCertif\DATA\processed\life_quality.csv", index=False)

In [18]:
df_life_quality.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 11 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Classement                             83 non-null     str    
 1   Pays                                   83 non-null     str    
 2   Indice de Qualité de Vie               83 non-null     float64
 3   Indice du Pouvoir d'Achat              83 non-null     float64
 4   Indice de Sécurité                     83 non-null     float64
 5   Indice des soins de santé              83 non-null     float64
 6   Indice du Coût de la Vie               83 non-null     float64
 7   Rapport Prix de l'immobilier/Revenu    83 non-null     float64
 8   Indice du Temps de Trajet dans Trafic  83 non-null     float64
 9   Indice de pollution                    83 non-null     float64
 10  Indice Climatique                      83 non-null     float64
dtypes: float64(9), str(

In [19]:
df_index_pays.info()

<class 'pandas.DataFrame'>
Index: 220 entries, 0 to 221
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   pays            220 non-null    str  
 1   code_2L         220 non-null    str  
 2   code_3L         220 non-null    str  
 3   code_numerique  220 non-null    str  
dtypes: str(4)
memory usage: 8.6 KB
